# UC4 - Synthesizing Executive Business Intelligence

## 1) Imports and configuration

In [0]:
import json
import re
from datetime import datetime, timezone
from openai import OpenAI
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

GOLD = "globalmart.gold"
MV_REVENUE = f"{GOLD}.agg_monthly_revenue_by_region"
MV_VENDOR_RET = f"{GOLD}.agg_vendor_return_rate"
MV_PROD_REG = f"{GOLD}.agg_product_region_monthly"
OUT_TABLE = f"{GOLD}.ai_business_insights"
MODEL_NAME = "databricks-gpt-oss-20b"

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_ws_raw = spark.conf.get("spark.databricks.workspaceUrl")
_ws_url = _ws_raw.replace("https://", "").replace("http://", "").strip("/")

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{_ws_url}/serving-endpoints",
)

INSIGHTS_SCHEMA = StructType(
    [
        StructField("insight_type", StringType(), True),
        StructField("executive_summary", StringType(), True),
        StructField("kpi_payload_json", StringType(), True),
        StructField("generated_at", TimestampType(), True),
    ]
)


## 2) Output table

In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {OUT_TABLE} (
      insight_type STRING,
      executive_summary STRING,
      kpi_payload_json STRING,
      generated_at TIMESTAMP
    ) USING DELTA
    COMMENT 'UC4 executive AI narratives generated from Gold KPI aggregates'
    """
)

DataFrame[]

## 3) Helpers: parse `databricks-gpt-oss-20b` structured output and call model

In [0]:
def _extract_text_from_blocks(obj) -> str:
    if obj is None:
        return ""
    if isinstance(obj, list):
        parts = []
        for block in obj:
            if not isinstance(block, dict):
                if isinstance(block, str) and block.strip():
                    parts.append(block.strip())
                continue
            if block.get("type") == "text":
                t = block.get("text")
                if isinstance(t, str) and t.strip():
                    parts.append(t.strip())
                elif isinstance(t, list):
                    parts.append(" ".join(str(x) for x in t if x is not None))
            elif "text" in block and block.get("text"):
                parts.append(str(block["text"]).strip())
        return "\n\n".join(parts) if parts else ""
    if isinstance(obj, dict):
        if obj.get("type") == "text" and "text" in obj:
            return str(obj["text"]).strip()
        if "text" in obj and obj["text"] is not None:
            return str(obj["text"]).strip()
    return ""


def extract_text_from_gpt_oss_content(raw) -> str:
    if raw is None:
        return ""
    if isinstance(raw, (list, dict)):
        out = _extract_text_from_blocks(raw)
        if out:
            return out
        if isinstance(raw, dict):
            return str(raw.get("content", raw)).strip()
        flat = [str(x).strip() for x in raw if x is not None and str(x).strip()]
        return "\n\n".join(flat) if flat else ""

    s = raw.strip() if isinstance(raw, str) else str(raw).strip()
    if not s:
        return ""

    try:
        parsed = json.loads(s)
        out = _extract_text_from_blocks(parsed)
        if out:
            return out
    except json.JSONDecodeError:
        pass

    for m in re.finditer(
        r'\{\s*"type"\s*:\s*"text"\s*,\s*"text"\s*:\s*"((?:[^"\\]|\\.)*)"\s*\}',
        s,
        re.DOTALL,
    ):
        try:
            return bytes(m.group(1), "utf-8").decode("unicode_escape").replace('\\"', '"')
        except Exception:
            return m.group(1).replace('\\"', '"')

    return s


def generate_executive_summary(insight_type: str, kpis: dict) -> str:
    system = (
        "You are a retail executive analytics assistant for GlobalMart. "
        "You will receive KPI aggregates only (not raw rows). "
        "Write exactly 4-6 sentences in executive tone. "
        "Include: what happened, likely drivers, risk/opportunity, and one concrete next action. "
        "Do not invent numbers. Use only KPI values provided."
    )
    user = (
        f"Insight type: {insight_type}\n"
        "KPI aggregate payload (JSON):\n"
        f"{json.dumps(kpis, ensure_ascii=False, indent=2)}\n\n"
        "Return only the narrative summary text."
    )

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=650,
        temperature=0.25,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )

    raw = ""
    if resp.choices and resp.choices[0].message:
        raw = resp.choices[0].message.content or ""
    return extract_text_from_gpt_oss_content(raw).strip()


## 4) Build KPI aggregates (no raw-row prompting)

In [0]:
mv_rev = spark.table(MV_REVENUE)
mv_vendor = spark.table(MV_VENDOR_RET)
mv_prod = spark.table(MV_PROD_REG)

# revenue_performance KPIs
rev_totals = mv_rev.agg(
    F.sum("total_sales").alias("company_total_sales"),
    F.sum("total_profit").alias("company_total_profit"),
    F.sum("order_count").alias("company_total_orders"),
    F.sum("total_quantity").alias("company_total_quantity"),
).collect()[0]

rev_by_region = (
    mv_rev.groupBy("region_name")
    .agg(
        F.sum("total_sales").alias("region_sales"),
        F.sum("total_profit").alias("region_profit"),
        F.sum("order_count").alias("region_orders"),
    )
    .withColumn(
        "profit_margin_pct",
        F.when(F.col("region_sales") > 0, (F.col("region_profit") / F.col("region_sales")) * 100.0),
    )
)

top_region_row = rev_by_region.orderBy(F.desc("region_sales")).limit(1).collect()[0]
worst_margin_row = rev_by_region.orderBy(F.asc("profit_margin_pct")).limit(1).collect()[0]

revenue_kpis = {
    "company_total_sales": round(float(rev_totals["company_total_sales"] or 0.0), 2),
    "company_total_profit": round(float(rev_totals["company_total_profit"] or 0.0), 2),
    "company_total_orders": int(rev_totals["company_total_orders"] or 0),
    "company_total_quantity": round(float(rev_totals["company_total_quantity"] or 0.0), 2),
    "top_region_by_sales": {
        "region_name": str(top_region_row["region_name"]),
        "region_sales": round(float(top_region_row["region_sales"] or 0.0), 2),
        "region_profit": round(float(top_region_row["region_profit"] or 0.0), 2),
    },
    "lowest_profit_margin_region": {
        "region_name": str(worst_margin_row["region_name"]),
        "profit_margin_pct": round(float(worst_margin_row["profit_margin_pct"] or 0.0), 2),
    },
    "region_breakdown": [
        {
            "region_name": str(r["region_name"]),
            "region_sales": round(float(r["region_sales"] or 0.0), 2),
            "region_profit": round(float(r["region_profit"] or 0.0), 2),
            "region_orders": int(r["region_orders"] or 0),
            "profit_margin_pct": round(float(r["profit_margin_pct"] or 0.0), 2),
        }
        for r in rev_by_region.orderBy(F.desc("region_sales")).collect()
    ],
}

# vendor_return_rate KPIs
vendor_totals = mv_vendor.agg(
    F.sum("vendor_gross_sales").alias("total_vendor_sales"),
    F.sum("vendor_refunds").alias("total_vendor_refunds"),
).collect()[0]

vendor_scored = mv_vendor.withColumn(
    "refund_to_sales_pct",
    F.when(F.col("vendor_gross_sales") > 0, F.col("refund_to_sales_ratio") * 100.0),
)

worst_vendor = vendor_scored.orderBy(F.desc("refund_to_sales_ratio")).limit(1).collect()[0]
best_vendor = vendor_scored.orderBy(F.asc("refund_to_sales_ratio")).limit(1).collect()[0]

high_risk_count = vendor_scored.filter(F.col("refund_to_sales_ratio") >= F.lit(0.20)).count()
vendor_count = vendor_scored.count()

vendor_kpis = {
    "total_vendor_sales": round(float(vendor_totals["total_vendor_sales"] or 0.0), 2),
    "total_vendor_refunds": round(float(vendor_totals["total_vendor_refunds"] or 0.0), 2),
    "portfolio_refund_rate_pct": round(
        (float(vendor_totals["total_vendor_refunds"] or 0.0) / max(float(vendor_totals["total_vendor_sales"] or 0.0), 1e-9)) * 100.0,
        2,
    ),
    "vendor_count": int(vendor_count),
    "high_risk_vendor_count_refund_rate_20pct_plus": int(high_risk_count),
    "highest_return_rate_vendor": {
        "vendor_id": str(worst_vendor["vendor_id"]),
        "vendor_name": str(worst_vendor["vendor_name"]),
        "refund_to_sales_pct": round(float(worst_vendor["refund_to_sales_pct"] or 0.0), 2),
    },
    "lowest_return_rate_vendor": {
        "vendor_id": str(best_vendor["vendor_id"]),
        "vendor_name": str(best_vendor["vendor_name"]),
        "refund_to_sales_pct": round(float(best_vendor["refund_to_sales_pct"] or 0.0), 2),
    },
    "top_5_highest_return_rate_vendors": [
        {
            "vendor_id": str(r["vendor_id"]),
            "vendor_name": str(r["vendor_name"]),
            "refund_to_sales_pct": round(float(r["refund_to_sales_pct"] or 0.0), 2),
            "vendor_gross_sales": round(float(r["vendor_gross_sales"] or 0.0), 2),
            "vendor_refunds": round(float(r["vendor_refunds"] or 0.0), 2),
        }
        for r in vendor_scored.orderBy(F.desc("refund_to_sales_ratio")).limit(5).collect()
    ],
}

# slow_moving_inventory KPIs
slow_base = (
    mv_prod.groupBy("region_name", "product_id")
    .agg(
        F.sum("total_sales").alias("region_product_sales"),
        F.sum("total_quantity").alias("region_product_qty"),
        F.sum("order_count").alias("region_product_orders"),
    )
)

w_region = Window.partitionBy("region_name").orderBy(F.asc("region_product_sales"), F.asc("region_product_qty"))
slow_ranked = slow_base.withColumn("sales_pct_rank_low", F.percent_rank().over(w_region)).withColumn(
    "slow_moving_flag", F.col("sales_pct_rank_low") <= F.lit(0.20)
)

slow_rows = slow_ranked.filter(F.col("slow_moving_flag") == F.lit(True))
slow_totals = slow_rows.agg(
    F.count("*").alias("slow_moving_product_region_pairs"),
    F.sum("region_product_sales").alias("slow_moving_sales_exposure"),
    F.sum("region_product_qty").alias("slow_moving_quantity_exposure"),
).collect()[0]

slow_by_region = (
    slow_rows.groupBy("region_name")
    .agg(
        F.count("*").alias("slow_count"),
        F.sum("region_product_sales").alias("slow_sales"),
        F.sum("region_product_qty").alias("slow_qty"),
    )
    .orderBy(F.desc("slow_sales"))
)

top_slow_region = slow_by_region.limit(1).collect()[0]

slow_kpis = {
    "slow_moving_product_region_pairs": int(slow_totals["slow_moving_product_region_pairs"] or 0),
    "slow_moving_sales_exposure": round(float(slow_totals["slow_moving_sales_exposure"] or 0.0), 2),
    "slow_moving_quantity_exposure": round(float(slow_totals["slow_moving_quantity_exposure"] or 0.0), 2),
    "highest_slow_inventory_region": {
        "region_name": str(top_slow_region["region_name"]),
        "slow_sales": round(float(top_slow_region["slow_sales"] or 0.0), 2),
        "slow_count": int(top_slow_region["slow_count"] or 0),
    },
    "region_slow_inventory_breakdown": [
        {
            "region_name": str(r["region_name"]),
            "slow_count": int(r["slow_count"] or 0),
            "slow_sales": round(float(r["slow_sales"] or 0.0), 2),
            "slow_qty": round(float(r["slow_qty"] or 0.0), 2),
        }
        for r in slow_by_region.collect()
    ],
    "sample_slow_products_top_10": [
        {
            "region_name": str(r["region_name"]),
            "product_id": str(r["product_id"]),
            "region_product_sales": round(float(r["region_product_sales"] or 0.0), 2),
            "region_product_qty": round(float(r["region_product_qty"] or 0.0), 2),
            "sales_pct_rank_low": round(float(r["sales_pct_rank_low"] or 0.0), 4),
        }
        for r in slow_rows.orderBy(F.asc("region_product_sales")).limit(10).collect()
    ],
}

## 5) Generate 3 executive summaries from KPI JSON and write to Gold

In [0]:
generated_at = datetime.now(timezone.utc)

insights_payload = [
    ("revenue_performance", revenue_kpis),
    ("vendor_return_rate", vendor_kpis),
    ("slow_moving_inventory", slow_kpis),
]

rows = []
for insight_type, kpi_payload in insights_payload:
    summary = generate_executive_summary(insight_type, kpi_payload)
    rows.append((insight_type, summary, json.dumps(kpi_payload, ensure_ascii=False), generated_at))

spark.createDataFrame(rows, schema=INSIGHTS_SCHEMA).write.format("delta").mode("append").saveAsTable(OUT_TABLE)

display(spark.table(OUT_TABLE).orderBy(F.desc("generated_at")))

insight_type,executive_summary,kpi_payload_json,generated_at
vendor_return_rate,"Vendor return activity for the period shows a portfolio refund rate of 7.57%, with total sales of $2,110,189.96 and refunds of $159,700.11 across four vendors. Johnsons Logistics (VEN04) leads with a 9.86% refund‑to‑sales ratio, driven by $12,629.59 in gross sales and $12,449.92 in refunds, while Seaborne Ltd. (VEN02) follows with a 7.54% rate on $1,210,262.82 in sales and $91,308.29 in refunds. The absence of any vendor exceeding a 20% refund threshold indicates moderate risk, yet the higher rates for Johnsons and Voyage Enterprises suggest potential quality or fulfillment issues. This presents an opportunity to negotiate stricter performance metrics with high‑return vendors and to implement targeted quality controls. Next action: schedule a joint review meeting with Johnsons Logistics and Voyage Enterprises to analyze root causes of their elevated return rates and agree on corrective action plans.","{""total_vendor_sales"": 2110189.96, ""total_vendor_refunds"": 159700.11, ""portfolio_refund_rate_pct"": 7.57, ""vendor_count"": 4, ""high_risk_vendor_count_refund_rate_20pct_plus"": 0, ""highest_return_rate_vendor"": {""vendor_id"": ""VEN04"", ""vendor_name"": ""Johnsons Logistics"", ""refund_to_sales_pct"": 9.86}, ""lowest_return_rate_vendor"": {""vendor_id"": ""VEN01"", ""vendor_name"": ""Velocity Logistics"", ""refund_to_sales_pct"": 6.75}, ""top_5_highest_return_rate_vendors"": [{""vendor_id"": ""VEN04"", ""vendor_name"": ""Johnsons Logistics"", ""refund_to_sales_pct"": 9.86, ""vendor_gross_sales"": 126265.79, ""vendor_refunds"": 12449.92}, {""vendor_id"": ""VEN03"", ""vendor_name"": ""Voyage Enterprises"", ""refund_to_sales_pct"": 7.81, ""vendor_gross_sales"": 350867.69, ""vendor_refunds"": 27392.58}, {""vendor_id"": ""VEN02"", ""vendor_name"": ""Seaborne Ltd."", ""refund_to_sales_pct"": 7.54, ""vendor_gross_sales"": 1210262.82, ""vendor_refunds"": 91308.29}, {""vendor_id"": ""VEN01"", ""vendor_name"": ""Velocity Logistics"", ""refund_to_sales_pct"": 6.75, ""vendor_gross_sales"": 422793.65, ""vendor_refunds"": 28549.32}]}",2026-03-27T09:52:11.416Z
slow_moving_inventory,"GlobalMart has identified 1,000 slow‑moving product‑region pairs, exposing $12,842.19 in sales and 3,854 units to inventory risk. The West region accounts for the highest exposure, with 297 slow items generating $4,432.32 in sales and 1,341 units in stock. Central, East, and South also show significant slow inventory, with 251, 252, and 200 items respectively, contributing $3,253.07, $3,035.49, and $2,121.31 in sales. The concentration of low‑sales products—many with sales below 1.5 and minimal quantity—signals potential markdown or clearance opportunities, but also increases carrying costs. To mitigate risk, we should prioritize a targeted clearance campaign for the top 10 slow products, starting with the West region’s OFF‑BI‑10003784, which has 1 unit and $1.34 sales. This action will reduce inventory exposure and free capital for higher‑margin items.","{""slow_moving_product_region_pairs"": 1000, ""slow_moving_sales_exposure"": 12842.19, ""slow_moving_quantity_exposure"": 3854.0, ""highest_slow_inventory_region"": {""region_name"": ""West"", ""slow_sales"": 4432.32, ""slow_count"": 297}, ""region_slow_inventory_breakdown"": [{""region_name"": ""West"", ""slow_count"": 297, ""slow_sales"": 4432.32, ""slow_qty"": 1341.0}, {""region_name"": ""Central"", ""slow_count"": 251, ""slow_sales"": 3253.07, ""slow_qty"": 1000.0}, {""region_name"": ""East"", ""slow_count"": 252, ""slow_sales"": 3035.49, ""slow_qty"": 895.0}, {""region_name"": ""South"", ""slow_count"": 200, ""slow_sales"": 2121.31, ""slow_qty"": 618.0}], ""sample_slow_products_top_10"": [{""region_name"": ""South"", ""product_id"": ""OFF-AP-10002906"", ""region_product_sales"": 0.44, ""region_product_qty"": 1.0, ""sales_pct_rank_low"": 0.0}, {""region_name"": ""South"", ""product_id"": ""OFF-BI-10003727"", ""region_produc

## 6) ai_query() demonstration #1 (regional revenue assessment)

In [0]:
%sql
WITH kpi AS (
  SELECT to_json(named_struct(
    'total_sales', round(sum(total_sales), 2),
    'total_profit', round(sum(total_profit), 2),
    'top_region', max_by(region_name, total_sales)
  )) AS payload
  FROM globalmart.gold.agg_monthly_revenue_by_region
)
SELECT
  payload,
  ai_query(
    'databricks-gpt-oss-20b',
    concat('Write a concise executive assessment in 2-3 sentences from this KPI JSON: ', payload)
  ) AS raw_ai_response
FROM kpi;

payload,raw_ai_response
"{""total_sales"":2110189.96,""total_profit"":169064.09,""top_region"":""West""}","Total sales reached $2.11 million with a profit of $169 thousand, reflecting a healthy margin of roughly 8%. The West region remains the top contributor, underscoring its strategic importance for future growth."


## 7) ai_query() demonstration #2 (vendor return pressure assessment)

In [0]:
%sql
WITH vendor_kpi AS (
  SELECT 
    to_json(
      named_struct(
        'portfolio_refund_rate_pct', round(100 * sum(vendor_refunds) / nullif(sum(vendor_gross_sales), 0), 2),
        'high_risk_vendor_count_20pct_plus', sum(CASE WHEN refund_to_sales_ratio >= 0.20 THEN 1 ELSE 0 END),
        'worst_vendor_name', max_by(vendor_name, refund_to_sales_ratio),
        'worst_vendor_refund_rate_pct', round(100 * max(refund_to_sales_ratio), 2)
      )
    ) AS payload
  FROM globalmart.gold.agg_vendor_return_rate
)
SELECT
  payload,
  ai_query(
    'databricks-gpt-oss-20b',
    concat('Write a concise executive assessment in 2-3 sentences from this KPI JSON: ', payload)
  ) AS raw_ai_response
FROM vendor_kpi;


payload,raw_ai_response
"{""portfolio_refund_rate_pct"":7.57,""high_risk_vendor_count_20pct_plus"":0,""worst_vendor_name"":""Johnsons Logistics"",""worst_vendor_refund_rate_pct"":9.86}","The portfolio’s overall refund rate stands at 7.57 %, comfortably below the 20 % threshold, and no vendors exceed that high‑risk level. However, Johnsons Logistics remains the outlier with a 9.86 % refund rate, indicating a potential area for improvement. Targeted engagement with Johnsons Logistics should help bring its performance in line with the broader portfolio."
